In [1]:
# NB06b_validation_protocol.ipynb

# %% [markdown]
# # NB06b – Leakage-Safe Validation Protocol
# **Purpose:** Replace plain stratified K-fold with a protocol that (i) keeps
# all records of the same firm in the same fold (GroupKFold) to prevent
# firm-level leakage across forecast horizons, (ii) reports the default ratio
# in every fold to document stratification, and (iii) adds a temporal
# hold-out so the model is tested on a period it did not train on.
# Addresses the reviewer concerns on leakage, fold balance, and time periods.
#
# **Requirements in the data:**
#   - a firm identifier column (to group records of the same firm)
#   - a forecast-horizon or year column (for the temporal split)
# If these do not exist, see Section 1 for fallbacks.
#
# **Input:**  `data/processed/override_data_naive.parquet`  (or b1/b2)
# **Output:**
#   - `results/tables/06b_fold_balance.csv`
#   - `results/tables/06b_validation_scores.csv`

# %%
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROC_DIR  = "../data/processed/"
TABLE_DIR = "../results/tables/"

# Choose which DGM's data to validate. For the main tables we recommend b1,
# since it is the leakage-safe, pd-independent design; naive is reported for
# contrast only.
df = pd.read_parquet(PROC_DIR + "override_data_b1.parquet")
print(f"Loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")

# %% [markdown]
# ## 1. Identify firm-id and horizon columns
# ADJUST these names to your data. Common possibilities are printed above.

# %%
FIRM_ID_COL  = None   # e.g. "firm_id", "company_id"  -> set if it exists
HORIZON_COL  = None   # e.g. "horizon", "year", "forecast_year"

# Fallbacks if identifiers are absent:
#   * No firm id  -> GroupKFold cannot group; we fall back to StratifiedKFold
#                    and state this limitation explicitly in the paper.
#   * No horizon  -> temporal split is skipped; only grouped CV is reported.
print("Set FIRM_ID_COL and HORIZON_COL above if these columns exist.")
print("If they do not exist in this dataset, tell me and we will document")
print("the limitation rather than fabricate a split.")

# %% [markdown]
# ## 2. Feature matrix
# We report TWO feature sets to separate the pd_system contribution from the
# override features, addressing the 'AUC is driven by pd_system' critique.

# %%
OVERRIDE_FEATS = ["pd_system", "grade_ordinal", "grade_diff", "override_flag"]
financial_feats = [c for c in df.columns
                   if c.startswith("Attr")
                   or c in ["net_profit_to_assets", "total_liabilities_to_assets",
                            "working_capital_to_assets",
                            "current_assets_to_short_liabilities",
                            "cash_to_current_liabilities",
                            "retained_earnings_to_assets", "ebit_to_assets",
                            "book_value_equity_to_liabilities",
                            "sales_to_assets", "equity_to_assets"]]
financial_feats = [c for c in financial_feats if c not in OVERRIDE_FEATS]

FEATURE_SETS = {
    "financial_only"      : financial_feats,
    "financial_plus_ovr"  : financial_feats + ["grade_diff", "override_flag"],
    "full_incl_pd"        : financial_feats + OVERRIDE_FEATS,
}
y = df["default"].astype(int).values

def make_model():
    neg, pos = (y == 0).sum(), (y == 1).sum()
    return XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=neg / pos, eval_metric="auc",
        random_state=42, n_jobs=-1,
    )

# %% [markdown]
# ## 3. Grouped (leakage-safe) cross-validation with fold-balance reporting

# %%
def run_cv(feature_cols, groups=None, n_splits=5):
    X = df[feature_cols].values
    if groups is not None:
        splitter = GroupKFold(n_splits=n_splits)
        split_iter = splitter.split(X, y, groups=groups)
        cv_type = "GroupKFold"
    else:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True,
                                   random_state=42)
        split_iter = splitter.split(X, y)
        cv_type = "StratifiedKFold(fallback)"

    aucs, aps, balances = [], [], []
    for k, (tr, te) in enumerate(split_iter):
        m = make_model()
        m.fit(X[tr], y[tr])
        p = m.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(y[te], p))
        aps.append(average_precision_score(y[te], p))
        balances.append({
            "fold": k,
            "train_default_rate": round(y[tr].mean(), 4),
            "test_default_rate" : round(y[te].mean(), 4),
            "n_train": len(tr), "n_test": len(te),
        })
    return cv_type, np.array(aucs), np.array(aps), pd.DataFrame(balances)

groups = df[FIRM_ID_COL].values if FIRM_ID_COL else None

score_rows, balance_frames = [], []
for fs_name, cols in FEATURE_SETS.items():
    cv_type, aucs, aps, bal = run_cv(cols, groups=groups)
    score_rows.append({
        "feature_set": fs_name, "cv_type": cv_type,
        "roc_auc_mean": round(aucs.mean(), 4), "roc_auc_std": round(aucs.std(), 4),
        "pr_auc_mean" : round(aps.mean(), 4),  "pr_auc_std" : round(aps.std(), 4),
    })
    bal["feature_set"] = fs_name
    balance_frames.append(bal)

scores = pd.DataFrame(score_rows)
balance = pd.concat(balance_frames, ignore_index=True)

scores.to_csv(TABLE_DIR + "06b_validation_scores.csv", index=False)
balance.to_csv(TABLE_DIR + "06b_fold_balance.csv", index=False)

print("=== Validation scores by feature set ===")
print("Compare financial_only vs financial_plus_ovr to isolate the pure")
print("incremental contribution of override features (WITHOUT pd_system).")
print("full_incl_pd shows how much pd_system alone inflates the AUC.\n")
print(scores.to_string(index=False))

print("\n=== Fold balance (default rate per fold) ===")
print(balance.to_string(index=False))

Loaded: (43405, 75)
Columns: ['net_profit_to_assets', 'total_liabilities_to_assets', 'working_capital_to_assets', 'current_assets_to_short_liabilities', 'cash_to_current_liabilities', 'retained_earnings_to_assets', 'ebit_to_assets', 'book_value_equity_to_liabilities', 'sales_to_assets', 'equity_to_assets', 'Attr11', 'Attr12', 'Attr13', 'Attr14', 'Attr15', 'Attr16', 'Attr17', 'Attr18', 'Attr19', 'Attr20', 'Attr21', 'Attr22', 'Attr23', 'Attr24', 'Attr25', 'Attr26', 'Attr27', 'Attr28', 'Attr29', 'Attr30', 'Attr31', 'Attr32', 'Attr33', 'Attr34', 'Attr35', 'Attr36', 'Attr37', 'Attr38', 'Attr39', 'Attr40', 'Attr41', 'Attr42', 'Attr43', 'Attr44', 'Attr45', 'Attr46', 'Attr47', 'Attr48', 'Attr49', 'Attr50', 'Attr51', 'Attr52', 'Attr53', 'Attr54', 'Attr55', 'Attr56', 'Attr57', 'Attr58', 'Attr59', 'Attr60', 'Attr61', 'Attr62', 'Attr63', 'Attr64', 'class', 'year_horizon', 'default', 'pd_system', 'system_grade', 'grade_ordinal', 'override_flag', 'final_ordinal', 'final_grade', 'grade_diff', 'overri

In [2]:
# NB06b_horizon_check.ipynb

# %% [markdown]
# ## Check year_horizon semantics before attempting a temporal split
# If year_horizon encodes the forecast horizon (1-5 years ahead) rather than
# a calendar year, a chronological train/test split is not meaningful and we
# document firm-level/temporal leakage as a limitation instead.

# %%
print("year_horizon value counts:")
print(df["year_horizon"].value_counts().sort_index())
print("\nDefault rate by year_horizon:")
print(df.groupby("year_horizon")["default"].agg(n="count", dr="mean").round(4))
print("\n'class' value counts (for reference):")
print(df["class"].value_counts())

year_horizon value counts:
year_horizon
1     7027
2    10173
3    10503
4     9792
5     5910
Name: count, dtype: int64

Default rate by year_horizon:
                  n      dr
year_horizon               
1              7027  0.0386
2             10173  0.0393
3             10503  0.0471
4              9792  0.0526
5              5910  0.0694

'class' value counts (for reference):
class
0    41314
1     2091
Name: count, dtype: int64
